<a href="https://colab.research.google.com/github/sukanya9020/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sukanya9020/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

files = list(
    api.list_repo_tree(
        repo_id="FlyRank/internship-warehouse",
        repo_type="dataset",
        recursive=True
    )
)

for f in files[:50]:
    print(f.path)

fact_content_daily_performance
fact_content_daily_performance/month=2025-01
fact_content_daily_performance/month=2025-02
fact_content_daily_performance/month=2025-03
fact_content_daily_performance/month=2025-04
fact_content_daily_performance/month=2025-05
fact_content_daily_performance/month=2025-06
fact_content_daily_performance/month=2025-07
fact_content_daily_performance/month=2025-08
fact_content_daily_performance/month=2025-09
fact_content_daily_performance/month=2025-10
fact_content_daily_performance/month=2025-11
fact_content_daily_performance/month=2025-12
fact_content_daily_performance/month=2026-01
fact_content_daily_performance/month=2026-02
fact_content_daily_performance/month=2026-03
fact_content_daily_performance/month=2026-04
fact_content_daily_performance/month=2026-05
fact_content_daily_performance/month=2026-06
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/mon

In [ ]:
# Show the files inside the March 2026 daily-performance partition

march_files = [
    f.path
    for f in files
    if "fact_content_daily_performance/month=2026-03" in f.path
]

for path in march_files:
    print(path)

fact_content_daily_performance/month=2026-03
fact_content_daily_performance/month=2026-03/data_0.parquet


In [ ]:
# Find the actual Parquet file inside the March 2026 partition

march_parquet_files = [
    f.path
    for f in files
    if "fact_content_daily_performance/month=2026-03/" in f.path
    and f.path.endswith(".parquet")
]

print("March Parquet files:")
for path in march_parquet_files:
    print(path)

March Parquet files:
fact_content_daily_performance/month=2026-03/data_0.parquet


In [ ]:
from huggingface_hub import hf_hub_download

march_file = march_parquet_files[0]

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=march_file,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Downloaded:", march_path)

Downloaded: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [ ]:
import pandas as pd

march_df = pd.read_parquet(march_path)

print("Rows:", len(march_df))
print("\nColumns:")
print(march_df.columns.tolist())

Rows: 9841378

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. Data contract — Refresh / Content Opportunity Scoring

### 1) What does one row mean?

One row represents the daily performance of one content item for one pseudonymized client on one report date.

For this assignment, I use the March 2026 partition of `fact_content_daily_performance`.

### 2) Which table(s) will I use?

Primary table:

`fact_content_daily_performance`

I will use the March 2026 partition for the initial data-contract checks and feature exploration.

### 3) What is the time window?

The working window is March 2026.

I use this as a mid-panel month rather than the final June 2026 `_sample` month, because the assignment warns that the final month should be treated as a sealed outcome/test period.

### 4) What will I rank or predict?

I will rank content items by a refresh-opportunity score.

The score is intended to identify content that may deserve review or updating, using information that would be available at the decision moment. It is a prioritization proxy rather than a claim that a refresh will definitely improve performance.

### 5) One deliberate exclusion

I will deliberately exclude future outcome information and any label-derived/recommendation fields from the feature set.

These fields could reveal the answer instead of allowing the features to support an honest refresh recommendation.

### Decision moment

The decision moment is the point at which the available March information would be used to prioritize content for refresh review.

A valid feature must be knowable at that moment.

Any information that becomes available only after the decision moment, or directly describes the outcome being predicted, is excluded.

In [ ]:
print("Number of rows:", len(march_df))
print("\nColumn names:")
for i, col in enumerate(march_df.columns, start=1):
    print(f"{i}. {col}")

Number of rows: 9841378

Column names:
1. report_date
2. client_hash_id
3. content_hash_id
4. client_has_gsc
5. client_has_ga4
6. gsc_data_available
7. ga4_data_available
8. gsc_impressions
9. gsc_clicks
10. gsc_sum_position
11. gsc_avg_position
12. ga4_pageviews
13. ga4_sessions
14. ga4_users
15. ga4_engaged_sessions
16. ga4_total_engagement_sec
17. sessions_organic
18. sessions_direct
19. sessions_referral
20. sessions_social
21. sessions_paid
22. sessions_ai
23. ai_chatgpt
24. ai_perplexity
25. ai_gemini
26. ai_copilot
27. ai_claude
28. ai_meta
29. ai_other
30. scroll_events


In [ ]:
march_df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print("Total columns:", len(march_df.columns))

print("\nAll column names:")
for i, col in enumerate(march_df.columns, start=1):
    print(f"{i}. {col}")

Total columns: 30

All column names:
1. report_date
2. client_hash_id
3. content_hash_id
4. client_has_gsc
5. client_has_ga4
6. gsc_data_available
7. ga4_data_available
8. gsc_impressions
9. gsc_clicks
10. gsc_sum_position
11. gsc_avg_position
12. ga4_pageviews
13. ga4_sessions
14. ga4_users
15. ga4_engaged_sessions
16. ga4_total_engagement_sec
17. sessions_organic
18. sessions_direct
19. sessions_referral
20. sessions_social
21. sessions_paid
22. sessions_ai
23. ai_chatgpt
24. ai_perplexity
25. ai_gemini
26. ai_copilot
27. ai_claude
28. ai_meta
29. ai_other
30. scroll_events


In [ ]:
print(march_df.dtypes)

report_date                  object
client_hash_id               object
content_hash_id              object
client_has_gsc                 bool
client_has_ga4                 bool
gsc_data_available             bool
ga4_data_available           object
gsc_impressions               int64
gsc_clicks                    int64
gsc_sum_position              int64
gsc_avg_position            float64
ga4_pageviews               float64
ga4_sessions                float64
ga4_users                   float64
ga4_engaged_sessions        float64
ga4_total_engagement_sec    float64
sessions_organic            float64
sessions_direct             float64
sessions_referral           float64
sessions_social             float64
sessions_paid               float64
sessions_ai                 float64
ai_chatgpt                  float64
ai_perplexity               float64
ai_gemini                   float64
ai_copilot                  float64
ai_claude                   float64
ai_meta                     

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

### Features

For the Refresh / Content Opportunity Scoring lane, I will use these five features:

1. `gsc_impressions` — number of Google Search Console impressions.
2. `gsc_clicks` — number of Google Search Console clicks.
3. `gsc_avg_position` — average search position.
4. `ga4_pageviews` — number of pageviews recorded in GA4.
5. `scroll_events` — number of recorded scroll events.

### Label

There is no direct field in the March 2026 daily performance data that states whether a content refresh was successful.

Therefore, no direct refresh-success label is used in this section. Refresh opportunity is treated as a prioritization problem rather than a guaranteed refresh outcome.

### Context

These fields provide identification, timing, and data-availability context:

- `report_date` — identifies the reporting date.
- `client_hash_id` — identifies the pseudonymized client.
- `content_hash_id` — identifies the pseudonymized content item.
- `client_has_gsc` — indicates whether the client has Google Search Console access.
- `client_has_ga4` — indicates whether the client has GA4 access.
- `gsc_data_available` — indicates whether GSC data is available.
- `ga4_data_available` — indicates whether GA4 data is available.

### Excluded

The following fields are excluded from the initial five-feature frame:

- `sessions_organic`
- `sessions_direct`
- `sessions_referral`
- `sessions_social`
- `sessions_paid`
- `sessions_ai`
- `ai_chatgpt`
- `ai_perplexity`
- `ai_gemini`
- `ai_copilot`
- `ai_claude`
- `ai_meta`
- `ai_other`

**Why excluded:** These fields are not needed for the initial refresh-opportunity feature frame. Excluding them keeps the feature set small and makes the first analysis easier to audit.

`client_hash_id`, `content_hash_id`, and `report_date` are also excluded as predictive features because they identify the observation or its time rather than directly describing content performance.

I will also exclude any future outcome information or label-derived fields because they could cause data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 3. Verify it with queries (grain, counts, missing values, windows)

The March 2026 partition is used as the mid-panel working slice. The following checks verify the expected grain, row count, date window, and data availability.

In [ ]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE VIEW march AS
SELECT *
FROM read_parquet('{march_path}')
""")

print("March 2026 data loaded successfully.")

March 2026 data loaded successfully.


In [ ]:
query_1 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(
        DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id
    ) AS distinct_grain_rows
FROM march
"""

con.sql(query_1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┐
│ total_rows │ distinct_grain_rows │
│   int64    │        int64        │
├────────────┼─────────────────────┤
│    9841378 │             9841378 │
└────────────┴─────────────────────┘

In [ ]:
query_2 = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM march
"""

con.sql(query_2)

┌───────────┬───────────────────┬──────────────────┐
│ row_count │ first_report_date │ last_report_date │
│   int64   │       date        │       date       │
├───────────┼───────────────────┼──────────────────┤
│   9841378 │ 2026-03-01        │ 2026-03-31       │
└───────────┴───────────────────┴──────────────────┘

In [ ]:
query_3 = """
SELECT
    COUNT(*) AS rows_after_availability_filter,
    COUNT(*) - COUNT(gsc_impressions) AS missing_gsc_impressions,
    COUNT(*) - COUNT(gsc_clicks) AS missing_gsc_clicks,
    COUNT(*) - COUNT(gsc_avg_position) AS missing_gsc_avg_position
FROM march
WHERE client_has_gsc IS TRUE
  AND gsc_data_available IS TRUE
"""

con.sql(query_3)

┌────────────────────────────────┬─────────────────────────┬────────────────────┬──────────────────────────┐
│ rows_after_availability_filter │ missing_gsc_impressions │ missing_gsc_clicks │ missing_gsc_avg_position │
│             int64              │          int64          │       int64        │          int64           │
├────────────────────────────────┼─────────────────────────┼────────────────────┼──────────────────────────┤
│                        3611061 │                       0 │                  0 │                        0 │
└────────────────────────────────┴─────────────────────────┴────────────────────┴──────────────────────────┘

### Verification summary

- Query 1 checks the expected grain of one report date, one client, and one content item.
- Query 2 checks the March 2026 row count and observed date window.
- Query 3 checks Google Search Console availability using `IS TRUE` and reports missing values for the main GSC features.

These checks verify the March 2026 data contract before creating the feature frame.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


 Fields: feature / label / context / excluded

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Five-feature frame

For the Refresh / Content Opportunity Scoring lane, I will use five features from the March 2026 data. Each feature is limited to information that should be available at the decision moment.

In [ ]:
feature_df = march_df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "scroll_events"
    ]
].copy()

print("Feature frame shape:", feature_df.shape)

feature_df.head()

Feature frame shape: (9841378, 8)


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,NaN,NaN


### Feature availability at the decision moment

1. **`gsc_impressions`**  
   Available at the decision moment because it represents the observed Google Search Console impressions recorded for the reporting period.

2. **`gsc_clicks`**  
   Available at the decision moment because it represents the observed Google Search Console clicks recorded for the reporting period.

3. **`gsc_avg_position`**  
   Available at the decision moment because it represents the observed average search position recorded for the reporting period.

4. **`ga4_pageviews`**  
   Available at the decision moment when GA4 data is available because it represents observed pageview activity.

5. **`scroll_events`**  
   Available at the decision moment when the corresponding analytics data has been collected because it represents observed user engagement.

In [ ]:
print("Number of features:", 5)

print("\nFeatures:")
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "scroll_events"
]

for feature in features:
    print("-", feature)

Number of features: 5

Features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- scroll_events


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Leakage trap

To demonstrate data leakage, I will intentionally create one label-derived column from the observed March performance data and include it in a simple classification experiment.

The purpose is only to demonstrate how information derived from the outcome can make a model appear unrealistically accurate.

After the experiment, the leaked column will be removed and will not be used in the final feature set.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Limitation

One limitation of this analysis is that the March 2026 slice represents only one month of daily performance data. A single-month slice may not represent longer-term content trends or seasonal behavior.

Therefore, the refresh-opportunity prioritization should be treated as a proxy and not as proof that refreshing a particular content item will improve its future performance.

### Self-check

- [x] Five plain-words contract answers
- [x] Fields sorted into feature, label, context, and excluded
- [x] Exactly three verification queries with outputs visible
- [x] Availability checked using `IS TRUE`
- [x] Five-feature frame created
- [x] Availability-at-decision-moment explanation added for each feature
- [x] Deliberate leakage experiment shown
- [x] Leaked feature removed
- [x] One named limitation documented